In [ ]:
import torch
import numpy as np


import matplotlib.pyplot as plt

_levi_cita_symbol = np.zeros((3,3,3))
_levi_cita_symbol[0, 1, 2] = 1
_levi_cita_symbol[1, 2, 0] = 1
_levi_cita_symbol[2, 0, 1] = 1
_levi_cita_symbol[0, 2, 1] = -1
_levi_cita_symbol[1, 0, 2] = -1
_levi_cita_symbol[2, 1, 0] = -1
_levi_cita_symbol = torch.tensor(_levi_cita_symbol, dtype=torch.float)


wavelength = 0.17
k = 2 * torch.pi / wavelength
th_approx = 0.3

intensity = []
offset = []

rocking_angle = torch.linspace(-0.01, 0.01, 201)

q_parallel = torch.linspace(-0.01, 0.01, 201)
psi = torch.stack((q_parallel, torch.zeros(201)), axis = 1)

data_matrix = torch.zeros((201, 201))

for irock, rock in enumerate(rocking_angle):

    p_vectors =  2 * k * np.sin(th_approx) * torch.Tensor([-np.sin(th_approx+rock), np.cos(th_approx+rock), 0.0])[None, :]
    xray_propagation_direction =  torch.Tensor([1.0, 0.0, 0.0])


    # Compute point of "exact bragg condition" in the plane Span(k_0, p)
    p_norm = torch.linalg.norm(p_vectors, axis=-1)
    theta_angle = torch.asin( p_norm / 2 / k )
    dir_scatteringplane_norm = (p_vectors - xray_propagation_direction[None, :] * np.einsum('xi,i->x', p_vectors, xray_propagation_direction)[:, None] )
    dir_scatteringplane_norm = dir_scatteringplane_norm / torch.linalg.norm(dir_scatteringplane_norm, axis=-1)[:, None]


    dir_outgoing = torch.cos(2 * theta_angle)[:, None] * xray_propagation_direction[None, :]\
        + torch.sin(2 * theta_angle)[:, None] * dir_scatteringplane_norm

    k_h_parallel = -torch.sin(2 * theta_angle)[:, None] * xray_propagation_direction[None, :]\
        + torch.cos(2 * theta_angle)[:, None] * dir_scatteringplane_norm
    k_h_orthogonal = torch.einsum('ijk,j,xk->xi', _levi_cita_symbol, xray_propagation_direction, dir_scatteringplane_norm)

    G_unit = torch.cos(theta_angle)[:, None]* dir_scatteringplane_norm - torch.sin(theta_angle)[:, None] * xray_propagation_direction[None, :]
    delta_q = (G_unit * 2 * k * torch.sin(theta_angle)[:, None] - p_vectors) / k


    # Make sample RMS and beam concentration_tensor 
    misorientation = 0.01
    strain_broadening = 1e-4

    S = (torch.eye(3) - torch.einsum('xi,xj->xij',G_unit,G_unit)) * (1/misorientation)**2  + (1/strain_broadening)**2 * torch.einsum('xi,xj->xij',G_unit,G_unit)

    divergence = 1e-4
    bandwidth = 1e-4

    B = (torch.eye(3) - np.outer(xray_propagation_direction, xray_propagation_direction))[None, :, :] * (1/divergence)**2  + (1/bandwidth)**2 * torch.outer(xray_propagation_direction, xray_propagation_direction)[None, :, :]




    W = torch.stack([k_h_parallel, k_h_orthogonal], axis=-1)
    T = torch.eye(3)[None, :, :] - torch.einsum('xi,j->xij', dir_outgoing, xray_propagation_direction)
    A = B + torch.einsum('xij,xjk,xkl->xil', T, S, T)
    E = torch.einsum('xij,xjk,xkl,xml,xmn->xin', S, T, torch.linalg.inv(A), T, S)
    F = torch.einsum('xia,xij,xjb->xab', W, S + E,  W)
    psi_zero = - torch.einsum('xab,xib,xij,xj->xa', torch.linalg.inv(F), W,  E-S, delta_q)
    H = torch.einsum('xi,xij,xj->x', psi_zero, F**2,  psi_zero)
    I = torch.einsum('xi,xij,xj->x', delta_q, S**2 - E**2,  delta_q)


    intens_term = 1 / torch.sqrt(torch.linalg.det(A)) * torch.exp(-I + H)
    divergence_concentration_tensor = torch.einsum('xia,xab,xjb->xij',W, torch.linalg.inv(F), W)



    argument = -torch.einsum('xa,ab,xb->x',psi-psi_zero, torch.linalg.inv(F[0]), psi-psi_zero)
    data_matrix[irock, :] = intens_term * torch.exp(argument)

    # dir_outgoing = dir_outgoing + psi_zero[:,0][:, None] * k_h_parallel\
    #     + psi_zero[:,1][:, None] * k_h_orthogonal





/tmp/ipykernel_396074/3031178048.py:33: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  p_vectors =  2 * k * np.sin(th_approx) * torch.Tensor([-np.sin(th_approx+rock), np.cos(th_approx+rock), 0.0])[None, :]
/tmp/ipykernel_396074/3031178048.py:40: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  dir_scatteringplane_norm = (p_vectors - xray_propagation_direction[None, :] * np.einsum('xi,i->x', p_vectors, xray_propagation_direction)[:, None] )
/tmp/ipykernel_396074/3031178048.py:68: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  B = (torch.eye(3) - np.outer(xray_propagation_direction, xray_propagation_direction))[None, :, :] * (1/divergence)**2  + (1/bandwidth)**2 * torch.outer(xray_propagation_direction, xray_propagation_d

tensor(0.0100)
tensor(-0.0100)
tensor(0.0099)
tensor(-0.0099)
tensor(0.0098)
tensor(-0.0098)
tensor(0.0097)
tensor(-0.0097)
tensor(0.0096)
tensor(-0.0096)
tensor(0.0095)
tensor(-0.0095)
tensor(0.0094)
tensor(-0.0094)
tensor(0.0093)
tensor(-0.0093)
tensor(0.0092)
tensor(-0.0092)
tensor(0.0091)
tensor(-0.0091)
tensor(0.0090)
tensor(-0.0090)
tensor(0.0089)
tensor(-0.0089)
tensor(0.0088)
tensor(-0.0088)
tensor(0.0087)
tensor(-0.0087)
tensor(0.0086)
tensor(-0.0086)
tensor(0.0085)
tensor(-0.0085)
tensor(0.0084)
tensor(-0.0084)
tensor(0.0083)
tensor(-0.0083)
tensor(0.0082)
tensor(-0.0082)
tensor(0.0081)
tensor(-0.0081)
tensor(0.0080)
tensor(-0.0080)
tensor(0.0079)
tensor(-0.0079)
tensor(0.0078)
tensor(-0.0078)
tensor(0.0077)
tensor(-0.0077)
tensor(0.0076)
tensor(-0.0076)
tensor(0.0075)
tensor(-0.0075)
tensor(0.0074)
tensor(-0.0074)
tensor(0.0073)
tensor(-0.0073)
tensor(0.0072)
tensor(-0.0072)
tensor(0.0071)
tensor(-0.0071)
tensor(0.0070)
tensor(-0.0070)
tensor(0.0069)
tensor(-0.0069)
tensor(0